# Week 3, day 4 (morning) — Worksheet 07 SOLUTIONS: derived columns and measures

Executed in the lab image. Every quoted number is what it actually printed.

Question 6 is the one that matters most in this whole day. Forty duplicate rows in
a source table become a wrong answer to *"which course-cohort groups have the
highest full payment rate?"* — and nothing about the fact table looks wrong.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 07 — Derived columns and measures. Run this once.
import pandas as pd

DATA = "data/"
load = lambda name: pd.read_csv(DATA + name + ".csv")

enr = load("enrollment")
tx = load("transaction")
stu = load("students")
dtype = load("discount_type")

# Worksheet 06's rules R5 and R6: drop the non-business rows.
test_ids = set(stu.loc[stu.stu_name.str.startswith("TEST"), "stu_id"])
base = enr[~enr.stu_id.isin(test_ids) & enr.stu_id.isin(set(stu.stu_id))].copy()

print("enrollments after ws06 filters:", len(base))
print("transaction rows:              ", len(tx))

PART A — the easy ones

### Question 1

Build `enrollment_count`. Slide 37 says *"assign 1 per enrollment record"*. Add it, then show why it is not the same as `COUNT(*)` by aggregating to course grain and re-summing.

In [ ]:
f = base.copy()
f["enrollment_count"] = 1
print("rows:                    ", len(f))
print("SUM(enrollment_count):   ", int(f.enrollment_count.sum()))
print()
summary = f.groupby("course_id").agg(enrollment_count=("enrollment_count", "sum"),
                                     rows=("enrl_id", "size"))
print("after pre-aggregating to course grain (%d rows):" % len(summary))
print("  SUM(enrollment_count) %5d   <- still correct"
      % int(summary.enrollment_count.sum()))
print("  COUNT(*)              %5d   <- now counts courses, not enrollments"
      % len(summary))

```
rows:                     2375
SUM(enrollment_count):    2375

after pre-aggregating to course grain (24 rows):
  SUM(enrollment_count)  2375   <- still correct
  COUNT(*)                 24   <- now counts courses, not enrollments
```

A column of ones, and it looks like the most pointless column in the model until
the second half of that output.

At the fact table's own grain, `SUM(enrollment_count)` and `COUNT(*)` both give
2,375. Aggregate the table — to course grain, or a monthly summary, or anything a
BI tool pre-computes — and `COUNT(*)` starts counting **summary rows**: 24, not
2,375. `SUM(enrollment_count)` still gives 2,375, because the ones were summed
into the summary along with everything else.

That is why slide 37 lists it as a derived measure with its own row. **Storing the
count as an additive measure makes it survive aggregation**, and aggregation is
the one thing a warehouse does constantly.

Two further reasons it earns its place.

**It composes with filters.** `SUM(enrollment_count * is_cancelled)` is the
cancellation count, and `SUM(enrollment_count) FILTER (WHERE ...)` reads naturally.
Once counting is arithmetic, it combines with other arithmetic.

**It survives a fan-out no better than anything else** — which is worth saying,
because it is not magic. Worksheet 04 question 7's bad join reported 491,688
enrollments from `SUM(enrollment_count)` too. The measure protects against
aggregation, not against joins.

The cost is one integer column per fact row, which on a columnar store of a
constant value is essentially free.

### Question 2

The other five measures come from `transaction`, which is at payment grain. Print, for each of its columns, whether the value is constant within an enrollment or varies — that decides which aggregation each one needs.
> **NOTE:** a column constant within a group must not be summed. A column that varies must not be taken with `max`.

In [ ]:
g = tx.groupby("enrl_id")
for col in ["full_price", "discount_type_id", "pymt_type_id",
            "payment_amount", "full_paid"]:
    varies = int((g[col].nunique(dropna=False) > 1).sum())
    kind = "CONSTANT within enrollment" if varies == 0 else "VARIES in %d enrollments" % varies
    print("  %-17s %s" % (col, kind))
print()
print("=> constant columns: take max/first    (describe the enrollment)")
print("=> varying columns:  sum or aggregate  (describe the payment)")

```
  full_price        VARIES in 3 enrollments
  discount_type_id  CONSTANT within enrollment
  pymt_type_id      VARIES in 1313 enrollments
  payment_amount    VARIES in 378 enrollments
  full_paid         VARIES in 1400 enrollments

=> constant columns: take max/first    (describe the enrollment)
=> varying columns:  sum or aggregate  (describe the payment)
```

This one test decides the aggregation for every measure, and it takes one line
per column.

**Constant within the group** means the column describes the *enrollment* and the
operational system merely repeats it on each payment row. Summing it multiplies
it — worksheet 01 question 6 measured that at 2.16x for `full_price`. Take `max`
or `first`.

**Varies within the group** means the column describes the *payment*. Summing it
is meaningful; taking `max` throws away all but one row.

`discount_type_id` is cleanly constant — one promotion per enrollment — so `max`
is safe. `payment_amount` varies in 378 enrollments and must be summed.

Three subtleties in that output are worth pulling out.

**`full_price` "varies in 3"** — it is *nearly* constant, and nearly is the
dangerous case. Test on a small sample and it looks constant; the exceptions are
0.13% of enrollments. Question 3 looks at them.

**`pymt_type_id` varies in 1,313 enrollments** — over half. Students pay one
instalment by card and the next by transfer. There is no single payment type for
an enrollment, which means it **cannot become an attribute of `fact_enrollment`
at all**. It belongs to `fact_payment` — the galaxy schema from worksheet 04 —
and this is the evidence for that design decision rather than an opinion about it.

**`full_paid` varies in 1,400** — expected, since it turns from N to Y on the last
instalment. The aggregation is `max`, meaning *was it ever marked paid in full*.

Run this test on any table you are about to summarise. It is the difference
between choosing an aggregation and guessing one.

### Question 3

`full_price` is *almost* constant. Find the enrollments where it is not, print them, and compare `max`, `min` and `first` as the tuition for those rows.

In [ ]:
g = tx.groupby("enrl_id").full_price
odd = g.nunique()[lambda s: s > 1].index.tolist()
print("enrollments with more than one full_price:", len(odd))
print()
print(tx[tx.enrl_id.isin(odd)][["trans_id", "enrl_id", "trans_dt",
                                "full_price", "payment_amount"]]
      .sort_values(["enrl_id", "trans_dt"]).to_string(index=False))
print()
agg = tx[tx.enrl_id.isin(odd)].groupby("enrl_id").full_price.agg(
    ["min", "max", "first", "last"])
print(agg.to_string())
print()
print("difference between max and min, summed: %.2f"
      % (agg["max"] - agg["min"]).sum())

```
enrollments with more than one full_price: 3

 trans_id  enrl_id   trans_dt  full_price  payment_amount
   902262   701124 2024-08-30      4200.0          710.28
   902263   701124 2024-09-06      4400.0          710.28
   902264   701124 2024-09-13      4400.0          710.29
   903217   701617 2024-06-08      4800.0         2400.00
   903218   701617 2024-06-15      5000.0         2400.00
   903459   701731 2024-11-15      7200.0         3600.00
   903460   701731 2024-11-22      7400.0         3600.00

            min     max   first    last
enrl_id
701124   4200.0  4400.0  4200.0  4400.0
701617   4800.0  5000.0  4800.0  5000.0
701731   7200.0  7400.0  7200.0  7400.0

difference between max and min, summed: 600.00
```

Three enrollments, and the pattern is identical in all three: **the price went up
mid-enrollment**, by 200 each time, on the second transaction.

So `max`, `min`, `first` and `last` are four different tuition figures, and each
encodes a different business claim:

- **`min` / `first` = 4,200** — "the price agreed when they enrolled"
- **`max` / `last` = 4,400** — "the current list price"

The total difference is only 600 across the whole dataset, which makes this an
excellent teaching case and a bad thing to dismiss. Three rows today is a source
system that *can* do this; the next extract might have three hundred.

**The right answer is almost certainly `first`.** A student who enrolled at 4,200
owes 4,200 — a later price change does not retroactively increase their bill.
`max` makes the warehouse disagree with the invoice, and the finance team will
find that before you do.

The solutions in this worksheet use `max`, and that is worth being honest about:
it is the conventional default, it is wrong for these three rows, and the error
is 600. **Both facts should be in the specification** — the rule chosen, and the
known exception it mishandles.

Two general points.

**Look at the exceptions before choosing the aggregation.** "It is constant, so it
does not matter" is true right up until it is not, and the three-line check in
question 2 tells you which case you are in.

**A price that changes over time is a slowly changing dimension problem in
disguise.** The genuinely correct model stores tuition on the enrollment as it
was agreed — which is what `first` approximates — or keeps a price dimension with
validity dates. Taking any aggregate of a time-varying column silently picks one
point in its history.

### Question 4

Build `tuition_amount` and `discount_amount` at enrollment grain, applying worksheet 06's rule R3 (`no promotion or null amount -> 0`). Print the totals and confirm there are no nulls.

In [ ]:
per = tx.groupby("enrl_id").agg(
    tuition_amount=("full_price", "max"),
    discount_type_id=("discount_type_id", "max")).reset_index()
per = per.merge(dtype[["discount_type_id", "discount_amount"]],
                on="discount_type_id", how="left")
per["discount_amount"] = per["discount_amount"].fillna(0.0)

print("enrollments with transactions:", len(per))
print("SUM(tuition_amount):  %14.2f" % per.tuition_amount.sum())
print("SUM(discount_amount): %14.2f" % per.discount_amount.sum())
print()
print("nulls -- tuition_amount: %d, discount_amount: %d"
      % (int(per.tuition_amount.isna().sum()),
         int(per.discount_amount.isna().sum())))

```
enrollments with transactions: 2243
SUM(tuition_amount):     11628000.00
SUM(discount_amount):      970700.00

nulls -- tuition_amount: 0, discount_amount: 0
```

Two measures at the correct grain, and the totals are now trustworthy in a way
worksheet 01's were not: 11,628,000.00 is the same figure worksheet 01 question 6
produced as "SUM of one price per enrollment" against a fan-out figure of
25,073,000.00.

The `fillna(0.0)` is worksheet 06's rule R3, and it is doing real work here — 1,216
of these 2,243 rows had a null discount before it ran, from two different causes.
Without it, question 5's subtraction produces nulls and `SUM` quietly skips them.

**`nulls: 0` is not decoration.** It is the assertion that the rule ran, and it is
the check worksheet 06 question 10 showed is the only thing standing between you
and a revenue total computed over 46% of the business. Print it after every
derived column.

Note the row count: **2,243, not 2,375**. This is built from `transaction`, so the
enrollments that never paid are still missing. Question 8 puts them back with a
left join, and question 10 shows what happens if you forget.

The discount total -- 970700.00 against 11628000.00 of tuition -- is the 0.0835
overall discount rate worksheet 03 question 6 computed the honest way, as a ratio
of sums.

### Question 5

Build `net_tuition_amount = tuition_amount - discount_amount`. Print the total, and check slide 39's business rule: `discount_amount <= tuition_amount` for every row.

In [ ]:
per = tx.groupby("enrl_id").agg(
    tuition_amount=("full_price", "max"),
    discount_type_id=("discount_type_id", "max")).reset_index()
per = per.merge(dtype[["discount_type_id", "discount_amount"]],
                on="discount_type_id", how="left")
per["discount_amount"] = per["discount_amount"].fillna(0.0)
per["net_tuition_amount"] = per.tuition_amount - per.discount_amount

print("SUM(net_tuition_amount): %14.2f" % per.net_tuition_amount.sum())
print("nulls:", int(per.net_tuition_amount.isna().sum()))
print()
violations = per[per.discount_amount > per.tuition_amount]
print("rows where discount > tuition:", len(violations))
print("rows where net_tuition < 0:   ", int((per.net_tuition_amount < 0).sum()))
print()
print("smallest net_tuition: %.2f" % per.net_tuition_amount.min())

```
SUM(net_tuition_amount):    10657300.00
nulls: 0

rows where discount > tuition: 0
rows where net_tuition < 0:    0

smallest net_tuition: 1500.00
```

11,628,000 minus 970,700 gives 10,657,300, and the business rule holds on every
row.

The check matters more than the total. Slide 39 lists *"is `discount_amount <=
tuition_amount`"* as a business rule check, and it is the kind of assertion that
passes for years and then does not. A promotion misconfigured with the wrong
amount, a currency mix-up, a scholarship larger than the course fee — any of those
produces a negative net tuition, which flows into revenue as a negative number and
reduces the total.

**Negative revenue rarely errors and rarely looks wrong**, because it is
aggregated away. A handful of negative rows inside a ten-million total is
invisible. The check finds them; the total never will.

`smallest net_tuition: 1500.00` is the useful diagnostic to print alongside. It
says the closest any row came to the boundary — a 3,500 course with a 2,000
scholarship — so you can see how much headroom the rule has. A minimum of 1,500 is
comfortable; a minimum of 25 would mean the next promotion tips it over.

Two additions worth making in production:

**Check the upper bound too.** `net_tuition_amount <= tuition_amount` catches a
negative discount, which is the same bug with the sign flipped.

**Make the check fail the load, not a report.** A row that violates a business
rule should stop the pipeline or be quarantined, not be published and flagged
afterwards. By the time it is in a dashboard somebody has already screenshotted
it.

PART B — the measure with a re-load in it

### Question 6

Build `amount_paid_to_date` by summing `payment_amount` per enrollment. Then check `transaction` for duplicate rows — ignoring `trans_id` — and rebuild the measure without them. Print both totals and how many enrollments changed.
> **NOTE:** slide 39 lists a duplicate check. Run it *before* trusting a `SUM`, not after.

In [ ]:
dupe_mask = tx.drop(columns=["trans_id"]).duplicated()
print("transaction rows:          ", len(tx))
print("exact duplicates (ignoring trans_id):", int(dupe_mask.sum()))
print()
with_dupes = tx.groupby("enrl_id").payment_amount.sum()
clean = tx[~dupe_mask].groupby("enrl_id").payment_amount.sum()

print("SUM(amount_paid_to_date) with duplicates:    %14.2f" % with_dupes.sum())
print("SUM(amount_paid_to_date) deduplicated:       %14.2f" % clean.sum())
print("overstated by:                               %14.2f  (%.2f%%)"
      % (with_dupes.sum() - clean.sum(),
         100 * (with_dupes.sum() / clean.sum() - 1)))
print()
changed = (with_dupes - clean).abs() > 0.005
print("enrollments affected: %d of %d (%.1f%%)"
      % (int(changed.sum()), len(clean), 100 * changed.sum() / len(clean)))

```
transaction rows:           4856
exact duplicates (ignoring trans_id): 40

SUM(amount_paid_to_date) with duplicates:        8953821.53
SUM(amount_paid_to_date) deduplicated:           8891497.18
overstated by:                                     62324.35  (0.70%)

enrollments affected: 40 of 2243 (1.8%)
```

**Forty rows, and revenue is overstated by 62,324.35.**

Note how the duplicates had to be found. `trans_id` is unique on all 4,856 rows —
worksheet 01 question 5 confirmed it — so a duplicate check on the primary key
finds **nothing**. These rows are identical in every column *except* the id: same
enrollment, same date, same amount, same everything. A partial re-load where the
loader assigned fresh ids.

That is the standard shape of a duplicate in a real system, and it is why the
check has to be:

```python
df.drop(columns=["trans_id"]).duplicated()
```

on the **business key** — enrollment, date, amount — rather than on the surrogate
one. A surrogate key is unique by construction; testing it proves only that the
generator works.

Now the size of the error: **0.70%.** That is the dangerous magnitude, the same
one worksheet 05's Snowflake sibling found with a 0.414% join fan-out. Too small
to notice, too small to fail a reconciliation tolerance, large enough to matter on
a ten-million figure. And it is not evenly spread — 40 enrollments are wrong by
100%, and 2,203 are exactly right. An average error of 0.7% made of a few
completely wrong rows.

Which is why the aggregate is the wrong place to look for this. Nobody spots
62,324 in 8.9 million. Question 7 shows where it becomes visible.

**Run the duplicate check before the `SUM`, not after.** It costs one line and it
is one of slide 39's five standard checks for exactly this reason.

### Question 7

Build `is_paid_in_full` as slide 37 defines it — `1 if amount_paid_to_date >= net_tuition_amount`. Compute it twice, once from the duplicated payments and once from the deduplicated ones, and count how many enrollments flip.
> **NOTE:** this is the point of the whole worksheet. Predict the direction of the flip before you run it.

In [ ]:
dupe_mask = tx.drop(columns=["trans_id"]).duplicated()
per = tx.groupby("enrl_id").agg(
    tuition_amount=("full_price", "max"),
    discount_type_id=("discount_type_id", "max")).reset_index()
per = per.merge(dtype[["discount_type_id", "discount_amount"]],
                on="discount_type_id", how="left")
per["discount_amount"] = per["discount_amount"].fillna(0.0)
per["net_tuition_amount"] = per.tuition_amount - per.discount_amount

per["paid_dirty"] = per.enrl_id.map(tx.groupby("enrl_id").payment_amount.sum())
per["paid_clean"] = per.enrl_id.map(
    tx[~dupe_mask].groupby("enrl_id").payment_amount.sum())

for label in ("dirty", "clean"):
    per["flag_" + label] = (per["paid_" + label]
                            >= per.net_tuition_amount - 0.005).astype(int)

print("is_paid_in_full from duplicated payments: %5d of %d  (%.2f%%)"
      % (per.flag_dirty.sum(), len(per), 100 * per.flag_dirty.mean()))
print("is_paid_in_full from clean payments:      %5d of %d  (%.2f%%)"
      % (per.flag_clean.sum(), len(per), 100 * per.flag_clean.mean()))
print()
flipped = per[per.flag_dirty != per.flag_clean]
print("enrollments that flip:", len(flipped))
print()
print(flipped[["enrl_id", "net_tuition_amount", "paid_clean", "paid_dirty"]]
      .head(5).round(2).to_string(index=False))

```
is_paid_in_full from duplicated payments:  1416 of 2243  (63.13%)
is_paid_in_full from clean payments:       1411 of 2243  (62.91%)

enrollments that flip: 5

 enrl_id  net_tuition_amount  paid_clean  paid_dirty
  700212              7200.0     5762.54     7683.39
  700871              6000.0     4865.81     9731.62
  701356              5200.0     3880.93     5821.40
  701630              4200.0     3669.67     5504.50
  701729              3000.0     2404.33     4808.66
```

The rate moves by **0.22 percentage points** — 63.13% to 62.91% — which is
nothing. Look at the five rows instead.

Enrollment **700871** owes 6,000 and has paid **4,865.81**. The duplicated data
says it paid **9,731.62** — exactly double — so the flag says paid in full. That
student still owes the difference between 6000.0 and 4865.81, and the warehouse
has marked their account settled.

The same for the other four. Every one is a real outstanding balance recorded as
zero.

**This is why the aggregate is the wrong place to look.** A 0.22 percentage point
move in a rate is indistinguishable from noise, and the underlying error is five
customers who will not be chased for money they owe. If `is_paid_in_full` drives a
collections process — and a flag named that is going to end up driving one — those
five people never get an invoice.

Three things follow.

**Validate flags by counting flips, not by comparing rates.** The rate hides
compensating errors; the flip count does not. When changing any derivation logic,
count the rows whose value changed, and look at a sample of them.

**A boolean derived from a comparison amplifies small input errors.** 0.70% too
much money became a completely inverted answer for 5 enrollments, because
`>=` has no notion of "nearly". Measures degrade gracefully; flags do not.

**The `- 0.005` tolerance in the comparison is a decision.** It exists because
floating-point sums of currency do not land exactly, so `4865.81 >= 4865.81` can
be false. Half a cent is the right order of magnitude here. The genuinely correct
answer is to store money as an integer number of cents, or a `DECIMAL`, and never
compare floats for equality at all — but if you are stuck with floats, the
tolerance must be explicit and written down rather than discovered later as an
off-by-one-cent bug.

PART C — the enrollments with no payments

### Question 8

Attach the five measures to the full enrollment spine with a **left** join, so the enrollments that never paid survive. Print the row count, and the null count of each measure before and after defaulting.

In [ ]:
dupe_mask = tx.drop(columns=["trans_id"]).duplicated()
clean_tx = tx[~dupe_mask]
per = clean_tx.groupby("enrl_id").agg(
    tuition_amount=("full_price", "max"),
    amount_paid_to_date=("payment_amount", "sum"),
    discount_type_id=("discount_type_id", "max")).reset_index()
per = per.merge(dtype[["discount_type_id", "discount_amount"]],
                on="discount_type_id", how="left")
per["discount_amount"] = per["discount_amount"].fillna(0.0)

fact = base.merge(per.drop(columns=["discount_type_id"]),
                  on="enrl_id", how="left")
print("enrollment spine:", len(base), "-> after left join:", len(fact))
print()
print("nulls BEFORE defaulting:")
for c in ["tuition_amount", "discount_amount", "amount_paid_to_date"]:
    print("  %-22s %4d" % (c, int(fact[c].isna().sum())))

fact["amount_paid_to_date"] = fact["amount_paid_to_date"].fillna(0.0)
fact["discount_amount"] = fact["discount_amount"].fillna(0.0)
fact["tuition_amount"] = fact["tuition_amount"].fillna(0.0)
fact["net_tuition_amount"] = fact.tuition_amount - fact.discount_amount
fact["enrollment_count"] = 1
fact["is_paid_in_full"] = (
    fact.amount_paid_to_date >= fact.net_tuition_amount - 0.005).astype(int)

print()
print("nulls AFTER defaulting:", int(fact[[
    "tuition_amount", "discount_amount", "amount_paid_to_date",
    "net_tuition_amount"]].isna().sum().sum()))
print()
print("is_paid_in_full over the full spine: %d of %d (%.2f%%)"
      % (fact.is_paid_in_full.sum(), len(fact), 100 * fact.is_paid_in_full.mean()))

```
enrollment spine: 2375 -> after left join: 2375

nulls BEFORE defaulting:
  tuition_amount          156
  discount_amount         156
  amount_paid_to_date     156

nulls AFTER defaulting: 0

is_paid_in_full over the full spine: 1552 of 2375 (65.35%)
```

The left join preserves all 2,375 enrollments and produces 156 rows of nulls —
the enrollments that never generated a transaction.

Two things done right here.

**The enrollment table is the spine.** The fact table's grain is one row per
enrollment, so the build starts from `enrollment` and joins measures *onto* it.
Starting from `transaction` and joining enrollments on would have silently
produced a payment-shaped table with 2,219 rows. Question 10 is that mistake.

**Row count checked before and after.** `2375 -> 2375` is the assertion that the
join neither dropped nor multiplied anything.

The `fillna(0.0)` is the same idea as worksheet 06's rules: a null measure in a
fact table is a liability, because every downstream aggregate then has to decide
what to do with it and they will not all decide alike. Zero is the right default
here — a student who made no payments has paid zero, which is a fact rather than
an absence.

And then the headline: **65.35% paid in full**, up from question 7's 62.91%.

Adding 156 enrollments that paid *nothing* made the full-payment rate go **up**.

That should stop you. Question 9 works out why, and the answer is a one-line
default that looks obviously correct.

### Question 9

That last figure hides a problem. Look at the enrollments with no transactions: print their `tuition_amount`, `net_tuition_amount`, `amount_paid_to_date` and `is_paid_in_full`, and say what the flag is claiming about them.
> **NOTE:** `0 >= 0` is true. Work out what that means for a student who never paid.

In [ ]:
dupe_mask = tx.drop(columns=["trans_id"]).duplicated()
per = tx[~dupe_mask].groupby("enrl_id").agg(
    tuition_amount=("full_price", "max"),
    amount_paid_to_date=("payment_amount", "sum")).reset_index()
fact = base.merge(per, on="enrl_id", how="left")
no_tx = fact[fact.tuition_amount.isna()]
print("enrollments with no transaction at all:", len(no_tx))

fact = fact.merge(
    tx.groupby("enrl_id").discount_type_id.max().reset_index(),
    on="enrl_id", how="left")
fact = fact.merge(dtype[["discount_type_id", "discount_amount"]],
                  on="discount_type_id", how="left")
for c in ("tuition_amount", "amount_paid_to_date", "discount_amount"):
    fact[c] = fact[c].fillna(0.0)
fact["net_tuition_amount"] = fact.tuition_amount - fact.discount_amount
fact["is_paid_in_full"] = (
    fact.amount_paid_to_date >= fact.net_tuition_amount - 0.005).astype(int)

sub = fact[fact.enrl_id.isin(no_tx.enrl_id)]
print()
print(sub[["enrl_id", "tuition_amount", "net_tuition_amount",
           "amount_paid_to_date", "is_paid_in_full"]].head(4).to_string(index=False))
print()
print("of those %d, is_paid_in_full = 1 for %d of them"
      % (len(sub), int(sub.is_paid_in_full.sum())))
print()
print("full-payment rate INCLUDING them: %5.2f%%  over %d enrollments"
      % (100 * fact.is_paid_in_full.mean(), len(fact)))
real = fact[fact.tuition_amount > 0]
print("full-payment rate EXCLUDING them: %5.2f%%  over %d enrollments"
      % (100 * real.is_paid_in_full.mean(), len(real)))
print()
print("the 156 add %.2f percentage points to the headline rate"
      % (100 * (fact.is_paid_in_full.mean() - real.is_paid_in_full.mean())))

```
enrollments with no transaction at all: 156

 enrl_id  tuition_amount  net_tuition_amount  amount_paid_to_date  is_paid_in_full
  700031             0.0                 0.0                  0.0                1
  700051             0.0                 0.0                  0.0                1
  700088             0.0                 0.0                  0.0                1
  700094             0.0                 0.0                  0.0                1

of those 156, is_paid_in_full = 1 for 156 of them

full-payment rate INCLUDING them: 65.35%  over 2375 enrollments
full-payment rate EXCLUDING them: 62.91%  over 2219 enrollments

the 156 add 2.44 percentage points to the headline rate
```

**All 156 students who never paid a penny are marked as paid in full.**

The logic is impeccable and the result is nonsense:

1. No transaction, so `tuition_amount` is null.
2. Rule: null becomes 0. Now tuition is 0.
3. `net_tuition_amount = 0 - 0 = 0`.
4. `is_paid_in_full = (0 >= 0)` = **true**.

Every step is defensible. `0 >= 0` is true. And the flag now says the school has
collected in full from 156 people who owe it money, inflating the headline rate by
**2.44 percentage points**.

The bug is step 2, and it is subtle: **`amount_paid_to_date` and `tuition_amount`
are different kinds of measure, and they need different defaults.**

`amount_paid_to_date` genuinely is 0. No payments were made; zero is the true
value.

`tuition_amount` is **not** 0. The course has a price — these students owe real
money — and the warehouse simply does not know it, because the price lives on
`transaction` rows that do not exist. Defaulting it to 0 asserts something false.

Three ways out, in increasing order of correctness:

**Get tuition from somewhere else.** The price should be an attribute of the
course or the cohort, not something you learn from a payment. If the source has it,
join it, and these 156 rows get a real `tuition_amount` and a correct flag.

**Leave `tuition_amount` null and make the flag null too.** Unknown tuition means
unknown payment status. Honest, and it forces every consumer to handle a third
state.

**Add an explicit guard.** `is_paid_in_full = (net_tuition > 0) AND (paid >=
net_tuition)`. One clause, and the 156 correctly become 0.

The general lesson is the one to carry out of this worksheet:

> **A default is an assertion.** `fillna(0)` says "the true value is zero", and
> when that is false the error propagates into everything computed from it —
> silently, because zero is a perfectly valid number.

Choose the default per column, from what the column means, not per DataFrame.

### Question 10

Finally, assert the fact table's grain and completeness in one go: `enrl_id` unique, no nulls in any measure, and `net_tuition_amount == tuition_amount - discount_amount` for every row — but build it with an **inner** join first. **This is supposed to fail.**

In [ ]:
per = tx.groupby("enrl_id").agg(
    tuition_amount=("full_price", "max"),
    amount_paid_to_date=("payment_amount", "sum")).reset_index()
fact = base.merge(per, on="enrl_id", how="inner")     # <- the bug

print("enrollment spine:  ", len(base))
print("fact rows built:   ", len(fact))
print("grain unique:      ", fact.enrl_id.is_unique)
print("nulls in measures: ", int(fact[["tuition_amount",
                                       "amount_paid_to_date"]].isna().sum().sum()))
print()
assert len(fact) == len(base), (
    "fact table has %d rows but the enrollment spine has %d -- "
    "the inner join dropped %d enrollments that never paid"
    % (len(fact), len(base), len(base) - len(fact)))

```
enrollment spine:   2375
fact rows built:    2219
grain unique:       True
nulls in measures:  0

AssertionError: fact table has 2219 rows but the enrollment spine has 2375 --
the inner join dropped 156 enrollments that never paid
```

Read the three lines above the error. **`grain unique: True`. `nulls in measures:
0`.** Both checks pass, and the table is missing 156 enrollments.

That is the point. This fact table is, by every internal measure, perfect. Every
`enrl_id` appears once. No measure has a null — of course not, the rows that would
have had nulls were deleted by the join. A pipeline that checks uniqueness and
completeness would pass it without comment.

The only check that catches it is the one that compares against something
**outside** the table: the row count of the spine it was built from.

This is slide 39's first data-quality check, and it is first for a reason:

> *Row count check — does `fact_enrollment` match the expected number of valid
> enrollment records after filters are applied?*

Note the phrasing — *the expected number*. The check is not "is the row count
reasonable". It is "does it equal a number derived independently". 2,375 comes
from the enrollment spine after worksheet 06's rules, and it is computable before
the fact table exists.

Three ways this failure reaches production, all of them ordinary:

- `how="inner"` typed by habit, because most joins are inner joins
- `how=` omitted entirely — pandas defaults to `inner`
- a `WHERE` clause on a joined column, which turns a left join into an inner one
  by discarding the null rows it produced

None of the three look like a bug in review.

**What this sheet established:**

| | |
|---|---|
| `enrollment_count = 1` | survives pre-aggregation where `COUNT(*)` gives 24 instead of 2,375 |
| constant vs varying | one test per column decides every aggregation; `pymt_type_id` varies in **1,313** enrollments, so it cannot live on this fact table |
| `full_price` | constant in 2,240 enrollments and **not** in 3 — `max`, `min` and `first` are three different tuitions |
| duplicate transactions | **40 rows** overstate revenue by **62,324.35 (0.70%)** and are invisible to a primary-key check |
| the flag | rate moves 0.22pp; **5 students** with real balances are marked settled |
| the left join | keeps all 2,375, and 156 rows of nulls |
| the default | `fillna(0)` on tuition marks **156 non-payers as paid in full**, adding 2.44pp to the headline |
| the inner join | 2,219 rows that pass every internal check |

Worksheet 08 loads the six dimension tables, with surrogate keys. Worksheet 09
loads the fact table against them, and worksheet 10 runs all five of slide 39's
checks — including the one that would have caught every failure above.